# GEDA -- Baseline Comparison Notebook

**Project:** Graph-Enhanced Differential Attention for Natural Disaster Detection

So sanh **2 baselines** tren **3 tasks** voi statistical tests:

| Model | Paper | Method |
|-------|-------|--------|
| Paper 1 (GNN) | Nascimento et al. 2025 | GraphSAGE-2L-Norm-Res + Late Fusion |
| Paper 2 (DiffAttn) | Munia et al. 2025 | CLIP + Differential Attention |
| GEDA (Ours) | Proposed | Graph + DiffAttn + Multi-task |

**Paper 1 protocol:** 4 few-shot sizes x 10 splits = 40 runs (Task 1 only)

**Paper 2 protocol:** 3 tasks x 5 seeds = 15 runs

---

# Phase 1: Environment Setup

In [ ]:
# === 1.1 Check GPU ===
import torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: Enable GPU in Runtime > Change runtime type')

In [ ]:
# === 1.2 Install Dependencies ===
import time
t0 = time.time()

!pip install -q torch-geometric
print(f'[1/2] PyG installed ({time.time()-t0:.0f}s)')

!pip install -q numpy pandas scikit-learn tqdm matplotlib pillow \
    scikit-image requests emoji umap-learn timm sentence-transformers \
    jsonlines psutil gdown transformers termcolor nltk imageio scipy
print(f'[2/2] ML deps installed ({time.time()-t0:.0f}s)')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
print(f'\nAll done! Total: {time.time()-t0:.0f}s')

In [ ]:
# === 1.3 Clone Repos + Setup Symlinks + Patches ===
import os

repos = [
    ('https://github.com/jdnascim/mm-class-for-disaster-data-with-gnn.git',
     '/content/mm-class-for-disaster-data-with-gnn'),
    ('https://github.com/Munia03/Multimodal_Crisis_Event.git',
     '/content/Multimodal_Crisis_Event'),
    ('https://github.com/Raghvendra-14/A-Multimodal-Approach-and-Dataset-to-Crisis-Summarization-in-Tweets.git',
     '/content/GEDA'),
]

for url, path in repos:
    if not os.path.isdir(path):
        !git clone --depth 1 {url} {path}
        print(f'Cloned: {os.path.basename(path)}')
    else:
        print(f'Exists: {os.path.basename(path)}')

# --- Setup symlink for GNN repo data/CrisisMMD_v2.0 ---
P1_DIR = '/content/mm-class-for-disaster-data-with-gnn'
DATASET_PATH = '/content/datasets/CrisisMMD_v2.0'
symlink_target = f'{P1_DIR}/data/CrisisMMD_v2.0'

if os.path.islink(symlink_target):
    os.unlink(symlink_target)
    print(f'Removed old symlink: {symlink_target}')

if not os.path.exists(symlink_target):
    os.makedirs(os.path.dirname(symlink_target), exist_ok=True)
    os.symlink(DATASET_PATH, symlink_target)
    print(f'Created symlink: {symlink_target} -> {DATASET_PATH}')
else:
    print(f'Symlink OK: {symlink_target}')

# Verify JSONL data splits exist
splits_dir = f'{P1_DIR}/data/CrisisMMD_v2.0_baseline_split/data_splits_ssl'
if os.path.isdir(splits_dir):
    n_splits = len([d for d in os.listdir(splits_dir) if os.path.isdir(f'{splits_dir}/{d}')])
    print(f'Data splits: {n_splits} directories in {splits_dir}')
else:
    print(f'WARNING: Data splits NOT found at {splits_dir}')
    print('  The GNN repo should contain the JSONL split files.')

# --- Patch 1: Fix emoji compatibility (emoji >= 2.0) ---
preprocess_path = f'{P1_DIR}/preprocess.py'
if os.path.exists(preprocess_path):
    with open(preprocess_path, 'r') as f:
        code = f.read()
    if 'get_emoji_regexp' in code:
        code = code.replace(
            'def remove_emoji(text):\n    return emoji.get_emoji_regexp().sub(u\'\', text)',
            'def remove_emoji(text):\n    try:\n        return emoji.replace_emoji(text, \'\')\n    except AttributeError:\n        return emoji.get_emoji_regexp().sub(u\'\', text)'
        )
        with open(preprocess_path, 'w') as f:
            f.write(code)
        print(f'Patched: preprocess.py (emoji >= 2.0 compatibility)')
    else:
        print(f'preprocess.py: already patched or compatible')
else:
    print(f'WARNING: {preprocess_path} not found')

# --- Patch 2: Add CLIP feature extraction (matching paper) ---
# Write CLIP code to a temp file to avoid escaping issues
clip_tmp = '/tmp/clip_features_patch.py'
with open(clip_tmp, 'w') as f:
    f.write('\n')
    f.write('# ======== CLIP Feature Extraction (Paper-matching) ========\n')
    f.write('# Added by GEDA baseline notebook to match paper description\n')
    f.write('from transformers import CLIPProcessor, CLIPModel\n')
    f.write('\n')
    f.write('def clip_image_features(gpu, sample=False):\n')
    f.write('    dfs = [None, None, None]\n')
    f.write('    if torch.cuda.is_available():\n')
    f.write('        dev = "cuda:" + str(gpu)\n')
    f.write('    else:\n')
    f.write('        dev = "cpu"\n')
    f.write('    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")\n')
    f.write('    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")\n')
    f.write('    model = model.to(dev)\n')
    f.write('    model = model.eval()\n')
    f.write('    for ix, split in enumerate(("train", "dev", "test")):\n')
    f.write('        data = [obj for obj in jsonlines.open(join(DATAPATH, split + ".jsonl"))]\n')
    f.write('        image_files = []\n')
    f.write('        text = []\n')
    f.write('        labels = []\n')
    f.write('        embedded_vectors = []\n')
    f.write('        for row in tqdm.tqdm(data):\n')
    f.write('            image_files.append(join(IMAGEPATH, row["image"]))\n')
    f.write('            text.append(row["text"])\n')
    f.write('            labels.append(row["label"])\n')
    f.write('        if sample is False:\n')
    f.write('            batch_size = 64\n')
    f.write('            for batch_start in tqdm.tqdm(range(0, len(image_files), batch_size)):\n')
    f.write('                batch_files = image_files[batch_start:batch_start+batch_size]\n')
    f.write('                batch_images = []\n')
    f.write('                for img_path in batch_files:\n')
    f.write('                    try:\n')
    f.write('                        img = Image.open(img_path).convert("RGB")\n')
    f.write('                        batch_images.append(img)\n')
    f.write('                    except Exception:\n')
    f.write('                        batch_images.append(Image.new("RGB", (224, 224)))\n')
    f.write('                inputs = processor(images=batch_images, return_tensors="pt", padding=True)\n')
    f.write('                pixel_values = inputs["pixel_values"].to(dev)\n')
    f.write('                with torch.no_grad():\n')
    f.write('                    vision_out = model.vision_model(pixel_values=pixel_values)\n')
    f.write('                    img_feats = model.visual_projection(vision_out.pooler_output)\n')
    f.write('                    img_feats = img_feats / img_feats.norm(p=2, dim=-1, keepdim=True)\n')
    f.write('                    embedded_vectors.extend(img_feats.cpu().numpy().tolist())\n')
    f.write('        else:\n')
    f.write('            embedded_vectors = torch.randn(len(image_files), 512).tolist()\n')
    f.write('        labels = [(1,0) if l == "not_informative" else (0,1) for l in labels]\n')
    f.write('        labels = np.array(labels)\n')
    f.write('        df = pd.DataFrame({"image_files": image_files,\n')
    f.write('                          "text": text,\n')
    f.write('                          "embeddings": [f for f in embedded_vectors],\n')
    f.write('                          "labels": np.argmax(labels, axis=1)})\n')
    f.write('        dfs[ix] = df\n')
    f.write('    return dfs\n')
    f.write('\n')
    f.write('\n')
    f.write('def clip_text_features(gpu, sample=False):\n')
    f.write('    dfs = [None, None, None]\n')
    f.write('    if torch.cuda.is_available():\n')
    f.write('        dev = "cuda:" + str(gpu)\n')
    f.write('    else:\n')
    f.write('        dev = "cpu"\n')
    f.write('    model = SentenceTransformer("clip-ViT-B-32-multilingual-v1", cache_folder=".")\n')
    f.write('    model.to(dev)\n')
    f.write('    model = model.eval()\n')
    f.write('    for ix, split in enumerate(("train", "dev", "test")):\n')
    f.write('        data = [obj for obj in jsonlines.open(join(DATAPATH, split + ".jsonl"))]\n')
    f.write('        labels = []\n')
    f.write('        clean_text = []\n')
    f.write('        text = []\n')
    f.write('        image_files = []\n')
    f.write('        for row in tqdm.tqdm(data):\n')
    f.write('            image_files.append(join(IMAGEPATH, row["image"]))\n')
    f.write('            text.append(row["text"])\n')
    f.write('            clean_text.append(preprocess.pre_process(\n')
    f.write('                row["text"], keep_hashtag=True, keep_special_symbols=True))\n')
    f.write('            labels.append(row["label"])\n')
    f.write('        if sample is False:\n')
    f.write('            tweets_emds = model.encode(clean_text, device=dev)\n')
    f.write('        else:\n')
    f.write('            tweets_emds = torch.randn(len(clean_text), 512)\n')
    f.write('        labels = [(1,0) if l == "not_informative" else (0,1) for l in labels]\n')
    f.write('        labels = np.array(labels)\n')
    f.write('        df = pd.DataFrame({"image_files": image_files,\n')
    f.write('                          "text": text,\n')
    f.write('                          "clean_tex": clean_text,\n')
    f.write('                          "embeddings": [f for f in tweets_emds],\n')
    f.write('                          "labels": np.argmax(labels, axis=1)})\n')
    f.write('        dfs[ix] = df\n')
    f.write('    return dfs\n')
print(f'Wrote CLIP patch to {clip_tmp}')

# Apply CLIP patch to feature_extraction.py
fe_path = f'{P1_DIR}/feature_extraction.py'
with open(fe_path, 'r') as f:
    fe_code = f.read()

# Remove old CLIP code if exists
marker = '# ======== CLIP Feature Extraction'
idx = fe_code.find(marker)
if idx > 0:
    fe_code = fe_code[:idx].rstrip() + '\n'
    print('Removed old CLIP code')

# Append clean CLIP code
with open(clip_tmp, 'r') as f:
    clip_code = f.read()
with open(fe_path, 'w') as f:
    f.write(fe_code + clip_code)
print('Patched: feature_extraction.py (CLIP functions added)')

# --- Patch 3: Add 'clip' to feature_fusion.py argparse + dispatch ---
ff_path = f'{P1_DIR}/feature_fusion.py'
with open(ff_path, 'r') as f:
    ff_code = f.read()

patched = False

old_img = "choices=['maxvit', 'mobilenet', 'mobilevit']"
new_img = "choices=['maxvit', 'mobilenet', 'mobilevit', 'clip']"
if old_img in ff_code:
    ff_code = ff_code.replace(old_img, new_img)
    patched = True

old_txt = "choices=['mpnet']"
new_txt = "choices=['mpnet', 'clip']"
if old_txt in ff_code and new_txt not in ff_code:
    ff_code = ff_code.replace(old_txt, new_txt)
    patched = True

if 'clip_text_features' not in ff_code:
    ff_code = ff_code.replace(
        'if args_cl.textft == "mpnet":\n    [df_text_train, df_text_dev, df_text_test] = mpnet_features(dev_id, args_cl.sample)',
        'if args_cl.textft == "mpnet":\n    [df_text_train, df_text_dev, df_text_test] = mpnet_features(dev_id, args_cl.sample)\nelif args_cl.textft == "clip":\n    [df_text_train, df_text_dev, df_text_test] = clip_text_features(dev_id, args_cl.sample)'
    )
    patched = True

if 'clip_image_features' not in ff_code:
    ff_code = ff_code.replace(
        'elif args_cl.imageft == "mobilevit":\n    [df_image_train, df_image_dev, df_image_test] = mobilevit_features(dev_id, args_cl.sample)',
        'elif args_cl.imageft == "mobilevit":\n    [df_image_train, df_image_dev, df_image_test] = mobilevit_features(dev_id, args_cl.sample)\nelif args_cl.imageft == "clip":\n    [df_image_train, df_image_dev, df_image_test] = clip_image_features(dev_id, args_cl.sample)'
    )
    patched = True

if patched:
    with open(ff_path, 'w') as f:
        f.write(ff_code)
    print('Patched: feature_fusion.py (CLIP dispatching added)')
else:
    print('feature_fusion.py: CLIP already present')

print('\nAll patches applied!')
print('  Image: openai/clip-vit-base-patch32 (CLIP ViT-B/32)')
print('  Text:  clip-ViT-B-32-multilingual-v1 (Multilingual CLIP)')


# Phase 2: Dataset Preparation

In [ ]:
# === 2.1 Download CrisisMMD v2.0 ===
import os, glob, zipfile, subprocess

DATASET_DIR = '/content/datasets'
DATASET_PATH = f'{DATASET_DIR}/CrisisMMD_v2.0'
URL = 'https://crisisnlp.qcri.org/data/crisismmd/CrisisMMD_v2.0.tar.gz'

os.makedirs(DATASET_DIR, exist_ok=True)

# Step 1: Download + extract tar.gz if needed
if not os.path.isdir(DATASET_PATH):
    TAR = f'{DATASET_DIR}/CrisisMMD_v2.0.tar.gz'
    if not os.path.exists(TAR) or os.path.getsize(TAR) < 1.8e9:
        print('Downloading CrisisMMD v2.0...')
        !wget -q --show-progress -c -O {TAR} {URL}
    print('Extracting tar.gz...')
    !cd {DATASET_DIR} && tar xzf CrisisMMD_v2.0.tar.gz
    print('tar.gz extracted!')
else:
    print(f'Dataset exists: {DATASET_PATH}')

# Step 2: Diagnostic - what's actually in the dataset dir?
print('\n--- Diagnostic ---')
print(f'Contents of {DATASET_PATH}:')
if os.path.isdir(DATASET_PATH):
    for item in sorted(os.listdir(DATASET_PATH)):
        full = os.path.join(DATASET_PATH, item)
        if os.path.isdir(full):
            n = len(os.listdir(full))
            print(f'  [DIR]  {item}/ ({n} items)')
        else:
            sz = os.path.getsize(full)
            print(f'  [FILE] {item} ({sz/1e6:.1f} MB)')

# Step 3: Find and unzip datasplit zip (may be nested)
split_dir = f'{DATASET_PATH}/crisismmd_datasplit_settingA'
if not os.path.isdir(split_dir):
    # Search for zip anywhere under datasets/
    zips = glob.glob(f'{DATASET_DIR}/**/crisismmd_datasplit_all.zip', recursive=True)
    if zips:
        print(f'\nFound zip: {zips[0]}')
        target_dir = os.path.dirname(zips[0])
        with zipfile.ZipFile(zips[0], 'r') as z:
            z.extractall(target_dir)
        print(f'Extracted to {target_dir}')
    else:
        print('\nNo datasplit zip found. Checking for nested CrisisMMD dir...')
        # Handle nested extraction: CrisisMMD_v2.0/CrisisMMD_v2.0/
        nested = f'{DATASET_PATH}/CrisisMMD_v2.0'
        if os.path.isdir(nested):
            print(f'Found nested dir: {nested}')
            # Move contents up
            for item in os.listdir(nested):
                src = os.path.join(nested, item)
                dst = os.path.join(DATASET_PATH, item)
                if not os.path.exists(dst):
                    os.rename(src, dst)
                    print(f'  Moved: {item}')
            # Try again
            zips = glob.glob(f'{DATASET_PATH}/crisismmd_datasplit_all.zip')
            if zips:
                with zipfile.ZipFile(zips[0], 'r') as z:
                    z.extractall(DATASET_PATH)
                print('Datasplit extracted!')
else:
    n_tsv = len(glob.glob(f'{split_dir}/*.tsv'))
    print(f'\nDatasplit OK: {split_dir} ({n_tsv} TSV files)')

# Step 4: Final verification
target_tsv = f'{split_dir}/task_informative_text_img_train.tsv'
if os.path.exists(target_tsv):
    print(f'\n[OK] Key file exists: task_informative_text_img_train.tsv')
else:
    print(f'\n[FAIL] Still missing: {target_tsv}')
    print('Please run: !find /content/datasets -name "*.tsv" | head -20')

imgs = glob.glob(f'{DATASET_PATH}/**/data_image/**/*.jpg', recursive=True)
if not imgs:
    imgs = glob.glob(f'{DATASET_DIR}/**/*.jpg', recursive=True)
print(f'Images found: {len(imgs)}')


In [ ]:
# === 2.2 Download LLaVA Descriptions (Paper 2) ===
import gdown, zipfile

test_tsv = f'{DATASET_PATH}/crisismmd_datasplit_settingA/task_damage_text_img_train.tsv'
has_llava = False
if os.path.exists(test_tsv):
    with open(test_tsv, 'r') as f:
        has_llava = 'LLaVA_text' in f.readline()

if has_llava:
    print('LLaVA data already present!')
else:
    print('Downloading LLaVA descriptions...')
    gdown.download(id='1xpYcshz9_KkQqw3tN9E86Df7UgmN6BaY', output='llava_data.zip')
    with zipfile.ZipFile('llava_data.zip', 'r') as z:
        z.extractall(DATASET_PATH)
    os.remove('llava_data.zip')
    print('LLaVA data extracted!')

In [ ]:
# === 2.3 Verify Dataset ===
import glob

print('=' * 50)
print('DATASET VERIFICATION')
print('=' * 50)

ann_dir = f'{DATASET_PATH}/crisismmd_datasplit_settingA'
for tsv in sorted(glob.glob(f'{ann_dir}/*.tsv')):
    with open(tsv) as f:
        lines = f.readlines()
        header = lines[0].strip() if lines else ''
    llava_mark = ' [+LLaVA]' if 'LLaVA_text' in header else ''
    print(f'  {os.path.basename(tsv)}: {len(lines)-1} samples{llava_mark}')

# Phase 3: Fix & Patch Paper Code

In [ ]:
# === 3.1 Patch Paper 2 (DiffAttn) Code ===
import os, glob

P2_DIR = '/content/Multimodal_Crisis_Event'

# --- 1. Fix paths.py ---
with open(f'{P2_DIR}/paths.py', 'w') as f:
    f.write("dataroot = '/content/datasets'\n")
print('1. paths.py fixed')

# --- 2. Robust line-by-line patch of crisismmd_dataset.py ---
with open(f'{P2_DIR}/crisismmd_dataset.py', 'r') as f:
    lines = f.readlines()

out = []
skip_next = 0  # number of continuation lines to skip
n_patched = 0

for i, line in enumerate(lines):
    s = line.strip()

    # Skip continuation lines from previously commented multi-line stmt
    if skip_next > 0:
        out.append('# [PATCHED] ' + line)
        skip_next -= 1
        n_patched += 1
        continue

    # tokenize_llamma body: 2-line statement
    if 'self.llamma_tokenizer(clean_text(' in s:
        out.append('# [PATCHED] ' + line)
        skip_next = 1  # skip the continuation line
        out.append('        ids = {}  # patched\n')
        n_patched += 1
        continue

    # tokenize_deepseek body: 2-line statement
    if 'self.deepseek_tokenizer(clean_text(' in s:
        out.append('# [PATCHED] ' + line)
        skip_next = 1  # skip the continuation line
        out.append('        ids = {}  # patched\n')
        n_patched += 1
        continue

    # llama init lines in initialize()
    if 'llama_model_name' in s and 'meta-llama' in s:
        out.append('# [PATCHED] ' + line)
        n_patched += 1
        continue

    if 'self.llamma_tokenizer = AutoTokenizer' in s:
        out.append('# [PATCHED] ' + line)
        n_patched += 1
        continue

    if 'self.llamma_tokenizer.pad_token' in s:
        out.append('# [PATCHED] ' + line)
        n_patched += 1
        continue

    # deepseek init lines
    if 'deepseek_model_name' in s and 'unsloth' in s:
        out.append('# [PATCHED] ' + line)
        n_patched += 1
        continue

    if 'self.deepseek_tokenizer = AutoTokenizer' in s:
        out.append('# [PATCHED] ' + line)
        n_patched += 1
        continue

    if 'self.deepseek_tokenizer.pad_token' in s:
        out.append('# [PATCHED] ' + line)
        n_patched += 1
        continue

    # llamma_tokens in read_data dict
    if "'llamma_tokens'" in s:
        out.append('# [PATCHED] ' + line)
        n_patched += 1
        continue

    out.append(line)

with open(f'{P2_DIR}/crisismmd_dataset.py', 'w') as f:
    f.writelines(out)
print(f'2. Patched {n_patched} lines in crisismmd_dataset.py')

# --- 3. Fix verbose in main.py ---
with open(f'{P2_DIR}/main.py', 'r') as f:
    c = f.read()
if 'verbose=True' in c:
    c = c.replace('verbose=True', '')
    with open(f'{P2_DIR}/main.py', 'w') as f:
        f.write(c)
    print('3. verbose removed')
else:
    print('3. verbose already clean')

# --- 4. Fix double CrisisMMD path ---
for py in glob.glob(f'{P2_DIR}/*.py'):
    with open(py, 'r') as f:
        c = f.read()
    if '/content/datasets/CrisisMMD_v2.0/CrisisMMD_v2.0' in c:
        c = c.replace('/content/datasets/CrisisMMD_v2.0/CrisisMMD_v2.0',
                      '/content/datasets/CrisisMMD_v2.0')
        with open(py, 'w') as f:
            f.write(c)
        print(f'4. Fixed path in {os.path.basename(py)}')

# --- 5. Syntax check ---

# --- 6. Fix TSV path only when split files are truly at dataset root ---
split_subdir = '/content/datasets/CrisisMMD_v2.0/crisismmd_datasplit_settingA'
root_probe = '/content/datasets/CrisisMMD_v2.0/task_informative_text_img_train.tsv'

with open(f'{P2_DIR}/crisismmd_dataset.py', 'r') as f:
    c = f.read()

if os.path.exists(root_probe) and (not os.path.isdir(split_subdir)) and 'crisismmd_datasplit_settingA/' in c:
    c = c.replace('crisismmd_datasplit_settingA/', '')
    with open(f'{P2_DIR}/crisismmd_dataset.py', 'w') as f:
        f.write(c)
    print('6. Fixed TSV path (using root-level TSV files)')
else:
    print('6. TSV path kept (using crisismmd_datasplit_settingA layout)')

import py_compile
try:
    py_compile.compile(f'{P2_DIR}/crisismmd_dataset.py', doraise=True)
    print('\n[OK] Syntax check PASSED!')
except py_compile.PyCompileError as e:
    print(f'\n[FAIL] Syntax error: {e}')

print('Paper 2 patched!')


# Phase 4: Run Paper 2 Baseline (DiffAttn)

**Protocol:** 3 tasks x 5 seeds = 15 runs

| Param | Value (from `train.sh` + `args.py`) |
|-------|-------------------------------------|
| Batch size | 32 |
| Learning rate | 0.001 (SGD, default in args.py) |
| Max iterations | 50 (from train.sh) |
| Text source | LLaVA (default in args.py) |
| Mode | both (image + text) |

In [ ]:
# === 4.0 Shared utilities ===
import time, json, os, csv

DRIVE_DIR = '/content/drive/MyDrive/CrisisSummarization'
RESULTS_CSV = '/content/geda_results/all_results.csv'

# Safe Drive setup for GEDA Results
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(f'{DRIVE_DIR}/geda_results', exist_ok=True)
    RESULTS_CSV = f'{DRIVE_DIR}/geda_results/all_results.csv'
    print(f'[SAFE MODE] GEDA results will map to: {RESULTS_CSV}')
else:
    os.makedirs('/content/geda_results', exist_ok=True)
    print(f'[WARNING] Saving locally to {RESULTS_CSV}')

def save_row(row_dict, csv_file=RESULTS_CSV):
    file_exists = os.path.isfile(csv_file)
    os.makedirs(os.path.dirname(csv_file), exist_ok=True)
    with open(csv_file, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row_dict.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)


In [ ]:
# === 4.1 Run Paper 2 -- Task 1 (Informative) ===
for seed in SEEDS:
    print(f'\n{"="*40}')
    print(f'  Task 1 | Seed {seed}')
    print(f'{"="*40}')
    
    model_name = f'task1_seed{seed}'
    t0 = time.time()
    
    %cd {P2_DIR}
    !python main.py \
        --model_name {model_name} \
        --mode both --task task1 \
        --batch_size 32 --device cuda \
        --max_iter 50 --text_from llava \
        --learning_rate 0.001 --num_workers 0
    
    elapsed = time.time() - t0
    metrics = parse_p2_log(model_name)
    result = {'model': 'paper2_diffattn', 'task': 'task1', 'seed': seed,
              'train_time_s': round(elapsed, 1), **metrics}
    save_row(result)
    print(f'  -> wF1={result.get("weighted_f1", "?")}, time={elapsed:.0f}s')

print('\nTask 1 complete!')

In [ ]:
# === 4.2 Run Paper 2 -- Task 2 (Humanitarian) ===
for seed in SEEDS:
    print(f'\n{"="*40}\n  Task 2 | Seed {seed}\n{"="*40}')
    model_name = f'task2_seed{seed}'
    t0 = time.time()
    
    %cd {P2_DIR}
    !python main.py \
        --model_name {model_name} \
        --mode both --task task2 \
        --batch_size 32 --device cuda \
        --max_iter 50 --text_from llava \
        --learning_rate 0.001 --num_workers 0
    
    elapsed = time.time() - t0
    metrics = parse_p2_log(model_name)
    result = {'model': 'paper2_diffattn', 'task': 'task2', 'seed': seed,
              'train_time_s': round(elapsed, 1), **metrics}
    save_row(result)
    print(f'  -> wF1={result.get("weighted_f1", "?")}')

print('\nTask 2 complete!')

In [ ]:
# === 4.3 Run Paper 2 -- Task 3 (Damage Severity) ===
for seed in SEEDS:
    print(f'\n{"="*40}\n  Task 3 | Seed {seed}\n{"="*40}')
    model_name = f'task3_seed{seed}'
    t0 = time.time()
    
    %cd {P2_DIR}
    !python main.py \
        --model_name {model_name} \
        --mode both --task task3 \
        --batch_size 32 --device cuda \
        --max_iter 50 --text_from llava \
        --learning_rate 0.001 --num_workers 0
    
    elapsed = time.time() - t0
    metrics = parse_p2_log(model_name)
    result = {'model': 'paper2_diffattn', 'task': 'task3', 'seed': seed,
              'train_time_s': round(elapsed, 1), **metrics}
    save_row(result)
    print(f'  -> wF1={result.get("weighted_f1", "?")}')

print('\nTask 3 complete!')

# Phase 5: Run Paper 1 Baseline (GNN)

## Setup: Paper-Matching CLIP Features
- **Paper:** CLIP ViT-B/32 (image) + Multilingual CLIP (text)
- **Code repo (modified):** MaxViT + MPNet (NOT matching paper)
- **Our setup:** Patched code to support CLIP, matching paper exactly

## Experiments

| Experiment | Image | Text | Fusion | PCA | Purpose |
|-----------|-------|------|--------|-----|--------|
| `paper_clip_pca` | **CLIP ViT-B/32** | **Multilingual CLIP** | Late | **256** | **Paper-matching** |
| `paper_clip_raw` | CLIP ViT-B/32 | Multilingual CLIP | Late | None | Ablation |
| `repo_maxvit_raw` | MaxViT tiny | MPNet | Late | None | Repo config |

## Common Params
| Param | Value |
|-------|-------|
| Architecture | `sage-2l-norm-res` (2x1024 + LN + Res) |
| Epochs/ES | 500 / 300 |
| Best model | `best_hm` (harmonic mean train+val F1) |
| L2 reg | lambda=1e-2, wd=1e-2 |
| Seed | 13 (hardcoded) |
| F1 metric | **Weighted F1** |
| Tasks | Task 1 only (Informativeness) |
| Few-shot | 50, 100, 250 (core) + 500 (optional) |
| Protocol | 4 sizes x 10 splits = 40 runs per experiment |

In [ ]:
# === 5.1 Run Paper 1 (GNN) â€” Paper-matching CLIP + exp_3/exp_5 ===
import time, json, os

P1_DIR = '/content/mm-class-for-disaster-data-with-gnn'
DATASET_PATH = './data/CrisisMMD_v2.0'
# Auto-detect Dataset in Drive
DRIVE_DIR = '/content/drive/MyDrive/CrisisSummarization'
if os.path.exists('/content/drive/MyDrive'):
    candidates = ['/content/drive/MyDrive/CrisisMMD_v2.0', '/content/drive/MyDrive/datasets/CrisisMMD_v2.0', f'{DRIVE_DIR}/CrisisMMD_v2.0']
    for c in candidates:
        if os.path.exists(c):
            # symlink the drive folder into ./data so Paper 1 script works natively
            os.makedirs('./data', exist_ok=True)
            if not os.path.exists(DATASET_PATH):
                os.symlink(c, DATASET_PATH)
            print(f'[SAFE MODE] Sourced Paper 1 Dataset from Drive')
            break
RUN_ID = 0
FEW_SHOT_SIZES = [50, 100, 250, 500]

# =====================================================
# EXPERIMENTS (exp_id must be unique to avoid collisions):
#   30 = CLIP + Late + PCA 256 (matches paper exactly)
#   50 = CLIP + Late + no PCA (ablation)
#    5 = MaxViT+MPNet + Late + no PCA (repo's exp_5)
#
# Code saves to: results/CrisisMMD/gnn/{exp_id}/{split}_{run}.json
# =====================================================
EXPERIMENTS = {
    'paper_clip_pca': {
        'imageft': 'clip', 'textft': 'clip',
        'reduction': '--reduction pca --pca_red 256',
        'exp_id': 30, 'model_name': 'paper1_gnn_clip_pca',
    },
    'paper_clip_raw': {
        'imageft': 'clip', 'textft': 'clip',
        'reduction': '',
        'exp_id': 50, 'model_name': 'paper1_gnn_clip_raw',
    },
    'repo_maxvit_raw': {
        'imageft': 'maxvit', 'textft': 'mpnet',
        'reduction': '',
        'exp_id': 5, 'model_name': 'paper1_gnn_maxvit_raw',
    },
}

# Choose which experiments to run:
# - ['paper_clip_pca'] = paper-matching only (recommended)
# - list(EXPERIMENTS.keys()) = all 3
RUNS_TO_DO = ['paper_clip_pca']

%cd {P1_DIR}

for run_key in RUNS_TO_DO:
    exp_cfg = EXPERIMENTS[run_key]
    EXP_ID = exp_cfg['exp_id']

    print(f'\n{"="*60}')
    print(f'  {run_key}: {exp_cfg["model_name"]}')
    print(f'  Features: image={exp_cfg["imageft"]}, text={exp_cfg["textft"]}')
    print(f'  Reduction: {exp_cfg["reduction"] or "None"}')
    print(f'  exp_id={EXP_ID} (results saved to results/CrisisMMD/gnn/{EXP_ID}/)')
    print(f'{"="*60}')

    total = len(FEW_SHOT_SIZES) * 10
    ct = 0

    for size in FEW_SHOT_SIZES:
        for split in range(10):
            ct += 1
            datasplit = f'{size}_s{split}'

            # Match GNN code: results/CrisisMMD/gnn/{exp_id}/{split}_{run_id}.json
            result_file = f'{P1_DIR}/results/CrisisMMD/gnn/{EXP_ID}/{datasplit}_{RUN_ID}.json'

            # Skip if already done
            if os.path.exists(result_file):
                with open(result_file) as f:
                    data = json.load(f)
                print(f'[{ct}/{total}] {datasplit} -> CACHED F1={data.get("f1_test", 0):.4f}')
                result = {
                    'model': exp_cfg['model_name'], 'task': 'task1',
                    'seed': split, 'few_shot': size,
                    'weighted_f1': data.get('f1_test', 0),
                    'balanced_accuracy': data.get('bacc_test', 0),
                    'train_time_s': 0,
                }
                save_row(result)
                continue

            print(f'\n[{ct}/{total}] {run_key}, size={size}, split={split}')
            t0 = time.time()

            reduction_args = exp_cfg['reduction']
            imageft = exp_cfg['imageft']
            textft = exp_cfg['textft']

            !python feature_fusion.py \
                --gpu_id 0 \
                --arch sage-2l-norm-res \
                --epochs 500 \
                --lr 1e-5 \
                --weight_decay 1e-2 \
                --dropout 0.5 \
                --n_neigh_train 16 \
                --n_neigh_full 16 \
                --lbl_train_frac 0.4 \
                --imagepath {DATASET_PATH} \
                --datasplit {datasplit} \
                --reg l2 \
                --l2_lambda 1e-2 \
                --exp_id {EXP_ID} \
                --imageft {imageft} \
                --textft {textft} \
                --fusion late \
                --loss nll \
                --shuffle_split \
                --early_stopping 300 \
                --best_model best_hm \
                --run_id {RUN_ID} {reduction_args} 2>&1 | tail -5

            elapsed = time.time() - t0

            if os.path.exists(result_file):
                with open(result_file) as f:
                    data = json.load(f)
                result = {
                    'model': exp_cfg['model_name'], 'task': 'task1',
                    'seed': split, 'few_shot': size,
                    'weighted_f1': data.get('f1_test', 0),
                    'balanced_accuracy': data.get('bacc_test', 0),
                    'train_time_s': round(elapsed, 1),
                }
                save_row(result)
                print(f'  -> F1={result["weighted_f1"]:.4f}, bAcc={result["balanced_accuracy"]:.4f} ({elapsed:.0f}s)')
            else:
                print(f'  -> FAILED')

    print(f'\n{run_key} complete! Total runs: {ct}')

print(f'\n{"="*60}')
print('All Paper 1 experiments complete!')
print(f'{"="*60}')


# Phase 6: Analysis & Comparison

In [ ]:
# === 6.1 Load All Results ===
import pandas as pd
import numpy as np

df = pd.read_csv('/content/geda_results/all_results.csv')

for col in ['accuracy', 'balanced_accuracy', 'micro_f1', 'macro_f1',
            'weighted_f1', 'train_time_s']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Total results: {len(df)}')
print(f'Models: {df["model"].unique()}')
print(f'Tasks:  {df["task"].unique()}')

# Paper 1 summary by few-shot size
p1 = df[df['model'].str.startswith('paper1_gnn')]
if len(p1) > 0 and 'few_shot' in p1.columns:
    print(f'\nPaper 1 few-shot sizes: {sorted(p1["few_shot"].dropna().unique())}')
    print(f'Paper 1 total runs: {len(p1)}')

# Paper 2 summary
p2 = df[df['model'] == 'paper2_diffattn']
if len(p2) > 0:
    print(f'\nPaper 2 seeds:  {sorted(p2["seed"].unique())}')
    print(f'Paper 2 total runs: {len(p2)}')

df.head(10)

In [ ]:
# === 6.2 Paper 1: Few-Shot Size Comparison ===
from scipy import stats

p1 = df[df['model'].str.startswith('paper1_gnn')].copy()

if len(p1) > 0:
    # Group by experiment variant
    for model_name in sorted(p1['model'].unique()):
        p1_variant = p1[p1['model'] == model_name]
        print(f'\n{"="*70}')
        print(f'  {model_name}')
        print(f'{"="*70}')
        print(f'{"Size":>6} {"wF1 Mean":>9} {"wF1 Std":>9} {"95% CI":>20} {"bAcc":>9} {"N":>3}')
        print(f'{"-"*6} {"-"*9} {"-"*9} {"-"*20} {"-"*9} {"-"*3}')

        for size in sorted(p1_variant['few_shot'].unique()):
            subset = p1_variant[p1_variant['few_shot'] == size]
            wf1 = subset['weighted_f1'].values
            bacc = subset['balanced_accuracy'].values

            if len(wf1) > 1:
                ci = stats.t.interval(0.95, len(wf1)-1, loc=np.mean(wf1), scale=stats.sem(wf1))
                ci_str = f'[{ci[0]*100:.1f}, {ci[1]*100:.1f}]'
            else:
                ci_str = 'N/A'

            ba_mean = np.mean(bacc)*100 if len(bacc) > 0 else 0
            tag = ' *' if size == 500 else ''
            print(f'{int(size):>6} {np.mean(wf1)*100:>9.2f}% {np.std(wf1, ddof=1)*100:>9.2f}%'
                  f' {ci_str:>20} {ba_mean:>9.2f}% {len(wf1):>3}{tag}')

        print(f'\n  NOTE: F1 = Weighted F1 (from sklearn f1_score average=weighted)')
        print(f'  NOTE: Seed = 13 (hardcoded in feature_fusion.py)')
        if 500 in p1_variant['few_shot'].values:
            print(f'  NOTE: * 500 samples is optional (Sirbu baseline comparison only)')
else:
    print('No Paper 1 results found yet.')


In [ ]:
# === 6.3 Cross-Model Comparison (Task 1 only -- both papers have it) ===
from scipy import stats

METRIC = 'weighted_f1'

print('=' * 70)
print('  TASK 1 (Informative) -- Cross-Model Comparison')
print('  Paper 1: auto-picks main GNN variant (prefers clip_pca, largest few-shot)')
print('  Paper 2: mean across 5 seeds')
print('=' * 70)

paper1_candidates = sorted([m for m in df['model'].dropna().unique() if str(m).startswith('paper1_gnn')])
paper1_main = 'paper1_gnn_clip_pca' if 'paper1_gnn_clip_pca' in paper1_candidates else (paper1_candidates[0] if paper1_candidates else None)

if paper1_main is None:
    p1_scores = np.array([])
    p1_label = 'Paper 1 (GNN)'
else:
    p1_df = df[df['model'] == paper1_main]
    few_sizes = sorted(p1_df['few_shot'].dropna().unique()) if 'few_shot' in p1_df.columns else []
    if len(few_sizes) > 0:
        best_size = int(few_sizes[-1])
        p1_scores = p1_df[p1_df['few_shot'] == best_size][METRIC].dropna().values
        p1_label = f'Paper 1 (GNN, {paper1_main}, N={best_size})'
    else:
        p1_scores = p1_df[METRIC].dropna().values
        p1_label = f'Paper 1 (GNN, {paper1_main})'

p2_scores = df[(df['model']=='paper2_diffattn') & (df['task']=='task1')][METRIC].dropna().values

for name, scores in [(p1_label, p1_scores),
                     ('Paper 2 (DiffAttn)', p2_scores)]:
    if len(scores) == 0:
        print(f'  {name}: No data')
        continue
    mean = np.mean(scores)
    std = np.std(scores, ddof=1) if len(scores) > 1 else 0
    if len(scores) > 1:
        ci = stats.t.interval(0.95, len(scores)-1, loc=mean, scale=stats.sem(scores))
        ci_str = f'[{ci[0]*100:.1f}, {ci[1]*100:.1f}]'
    else:
        ci_str = 'N/A'
    print(f'  {name:<45} {mean*100:>7.2f}% +/- {std*100:.2f}%  CI={ci_str}  N={len(scores)}')

# Significance test
if len(p1_scores) >= 2 and len(p2_scores) >= 2:
    min_n = min(len(p1_scores), len(p2_scores))
    t_stat, p_val = stats.ttest_ind(p1_scores[:min_n], p2_scores[:min_n])
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    diff = np.mean(p1_scores) - np.mean(p2_scores)
    print(f'\n  Independent t-test: t={t_stat:.3f}, p={p_val:.4f} ({sig})')
    print(f'  Mean difference: {diff*100:+.2f}%')


In [ ]:
# === 6.4 Paper 2: All Tasks Summary ===
TASKS = {'task1': 'Informative', 'task2': 'Humanitarian', 'task3': 'Damage Severity'}

print('=' * 70)
print('  PAPER 2 (DiffAttn) -- All Tasks (5 seeds each)')
print('=' * 70)
print(f'{"Task":<20} {"wF1 Mean":>10} {"wF1 Std":>10} {"Accuracy":>10} {"N":>4}')
print('-' * 60)

for task_id, task_name in TASKS.items():
    subset = df[(df['model']=='paper2_diffattn') & (df['task']==task_id)]
    wf1 = subset['weighted_f1'].dropna().values
    acc = subset['accuracy'].dropna().values
    if len(wf1) > 0:
        print(f'{task_name:<20} {np.mean(wf1)*100:>9.2f}% {np.std(wf1, ddof=1)*100:>9.2f}%'
              f' {np.mean(acc)*100:>9.2f}% {len(wf1):>3}')
    else:
        print(f'{task_name:<20} No data')

In [ ]:
# === 6.5 Visualization ===
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: Paper 1 few-shot curve ---
ax = axes[0]
p1 = df[df['model'].astype(str).str.startswith('paper1_gnn', na=False)]
sizes = sorted(p1['few_shot'].dropna().unique()) if len(p1) > 0 else []
means = [p1[p1['few_shot']==s]['weighted_f1'].dropna().mean()*100 for s in sizes]
stds = [p1[p1['few_shot']==s]['weighted_f1'].dropna().std()*100 for s in sizes]

if len(sizes) > 0:
    ax.errorbar([int(s) for s in sizes], means, yerr=stds, marker='o', capsize=5,
                color='#2196F3', linewidth=2, markersize=8, label='Paper 1 (GNN)')
    ax.set_xticks([int(s) for s in sizes])
    ax.legend(fontsize=10)
else:
    ax.text(0.5, 0.5, 'No Paper 1 data', ha='center', va='center', transform=ax.transAxes)

ax.set_xlabel('Few-Shot Size (labeled samples)', fontsize=12)
ax.set_ylabel('Weighted F1 (%)', fontsize=12)
ax.set_title('Paper 1: Semi-Supervised GNN\nFew-Shot Performance (Task 1)',
             fontsize=13, fontweight='bold')
ax.grid(alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# --- Right: Paper 2 all tasks ---
ax = axes[1]
TASKS = {'task1': 'Informative', 'task2': 'Humanitarian', 'task3': 'Damage'}
p2 = df[df['model'] == 'paper2_diffattn']
x_pos = range(len(TASKS))
p2_means = [p2[p2['task']==t]['weighted_f1'].dropna().mean()*100 for t in TASKS]
p2_stds = [p2[p2['task']==t]['weighted_f1'].dropna().std()*100 for t in TASKS]

bars = ax.bar(x_pos, p2_means, yerr=p2_stds, capsize=5,
              color='#FF5722', edgecolor='white', linewidth=1.5, alpha=0.85)
for bar, mean, std in zip(bars, p2_means, p2_stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.5,
            f'{mean:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Paper 2: Differential Attention\nAll Tasks (5 seeds)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Weighted F1 (%)', fontsize=12)
ax.set_xticks(x_pos)
ax.set_xticklabels(list(TASKS.values()), fontsize=10)
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.suptitle('GEDA Baseline Reproduction Results',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/geda_results/figures/baseline_comparison.png',
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved!')


In [ ]:
# === 6.6 LaTeX Table ===
METRIC = 'weighted_f1'
TASKS = {'task1': 'Informative', 'task2': 'Humanitarian', 'task3': 'Damage'}

print('% === Paper 1: Few-Shot Results (Task 1) ===')
print('\\begin{table}[h]')
print('\\centering')
print('\\caption{Paper 1 (GNN) Reproduction -- Weighted F1 by Few-Shot Size}')
print('\\begin{tabular}{c c c}')
print('\\hline')
print('\\textbf{N labeled} & \\textbf{Weighted F1 (\\%)} & \\textbf{Balanced Acc (\\%)} \\\\')
print('\\hline')

p1 = df[df['model'].astype(str).str.startswith('paper1_gnn', na=False)]
for size in sorted(p1['few_shot'].dropna().unique()):
    wf1 = p1[p1['few_shot']==size]['weighted_f1'].dropna().values
    bacc = p1[p1['few_shot']==size]['balanced_accuracy'].dropna().values
    wf1_str = f'{np.mean(wf1)*100:.1f} $\\pm$ {np.std(wf1, ddof=1)*100:.1f}' if len(wf1)>1 else f'{np.mean(wf1)*100:.1f}'
    bacc_str = f'{np.mean(bacc)*100:.1f} $\\pm$ {np.std(bacc, ddof=1)*100:.1f}' if len(bacc)>1 else '-'
    print(f'{int(size)} & {wf1_str} & {bacc_str} \\\\')

print('\\hline')
print('\\end{tabular}')
print('\\end{table}')

print('\n% === Paper 2: All Tasks ===')
print('\\begin{table}[h]')
print('\\centering')
print('\\caption{Paper 2 (DiffAttn) Reproduction -- Weighted F1 (\\%)}')
print('\\begin{tabular}{l c c c}')
print('\\hline')
print('\\textbf{Model} & \\textbf{Informative} & \\textbf{Humanitarian} & \\textbf{Damage} \\\\')
print('\\hline')

row = ['DiffAttn (reprod.)']
for task_id in TASKS:
    scores = df[(df['model']=='paper2_diffattn') & (df['task']==task_id)][METRIC].dropna().values
    if len(scores) > 0:
        row.append(f'{np.mean(scores)*100:.1f} $\\pm$ {np.std(scores, ddof=1)*100:.1f}')
    else:
        row.append('-')
print(' & '.join(row) + ' \\\\')
print('\\hline')
print('\\end{tabular}')
print('\\end{table}')


# Phase 7: GEDA Model â€” Training

Run the proposed GEDA model (Graph + DiffAttn + MTL) on same data.

**Protocol:** 5 seeds Ã— 3 tasks Ã— 4 few-shot sizes = 60 runs

In [ ]:
# === 7.1 Install GEDA dependencies ===
!pip install -q open_clip_torch faiss-cpu
print('GEDA dependencies installed')

In [ ]:
# === 7.2 Get GEDA code into Colab runtime ===
import sys, os

geda_files = ['src/models/geda_model.py', 'src/trainers/geda_trainer.py', 'geda_phase7_8.py']
missing = []

for f in geda_files:
    dst = f'/content/{f}'
    
    # Already exists in /content/?
    if os.path.exists(dst):
        print(f'OK: {dst}')
        continue
    
    # Try GEDA clone dir
    src = f'/content/GEDA/{f}'
    if os.path.exists(src):
        import shutil
        shutil.copy2(src, dst)
        print(f'Copied from GEDA clone: {dst}')
        continue
    
    # Try Google Drive
    drive_src = f'/content/drive/MyDrive/CrisisSummarization/{f}'
    if os.path.exists(drive_src):
        import shutil
        shutil.copy2(drive_src, dst)
        print(f'Copied from Drive: {dst}')
        continue
    
    missing.append(f)

# Upload missing files interactively
if missing:
    print(f'\nMissing: {missing}')
    print('Uploading via Colab file picker...')
    from google.colab import files
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(f'/content/{name}', 'wb') as fw:
            fw.write(content)
        print(f'Uploaded: /content/{name}')

if '/content' not in sys.path:
    sys.path.insert(0, '/content')

from src.models.geda_model import GEDAModel, model_summary
model = GEDAModel()
info = model_summary(model)
print(f'\nGEDA model: {info["trainable_params"]:,} trainable params')
print('GEDA code ready!')


In [ ]:
# === 7.3 Run ALL GEDA experiments ===
import os

# Use auto-detected Drive paths for safe execution of Phase 7
RESULTS_CSV = '/content/geda_results/all_results.csv'
RESULTS_DIR = '/content/geda_results'
DATASET_PATH = '/content/datasets/CrisisMMD_v2.0'

DRIVE_DIR = '/content/drive/MyDrive/CrisisSummarization'
if os.path.exists('/content/drive/MyDrive'):
    RESULTS_CSV = f'{DRIVE_DIR}/geda_results/all_results.csv'
    RESULTS_DIR = f'{DRIVE_DIR}/geda_results'
    candidates = ['/content/drive/MyDrive/CrisisMMD_v2.0', '/content/drive/MyDrive/datasets/CrisisMMD_v2.0', f'{DRIVE_DIR}/CrisisMMD_v2.0']
    for c in candidates:
        if os.path.exists(c):
            DATASET_PATH = c
            break

!python /content/geda_phase7_8.py \
    --dataset_path {DATASET_PATH} \
    --results_csv {RESULTS_CSV} \
    --output_dir {RESULTS_DIR} \
    --phase 7

print('\n=== Phase 7 complete! ===')


# Phase 8: 3-Model Comparison

Compare **Paper 1 (GNN)** vs **Paper 2 (DiffAttn)** vs **GEDA (Ours)**

Statistical tests: paired t-test + Bonferroni + Cohenâ€™s d

In [ ]:
# === 8.1 Unified 3-Model Comparison ===
import pandas as pd, numpy as np
from scipy import stats

df = pd.read_csv('/content/geda_results/all_results.csv')
METRIC = 'weighted_f1'
TASKS = ['task1', 'task2', 'task3']
TASK_NAMES = {'task1': 'Informativeness', 'task2': 'Humanitarian', 'task3': 'Damage'}

print('=' * 70)
print('  3-MODEL COMPARISON: Weighted F1 (mean +/- std)')
print('=' * 70)

for task_id in TASKS:
    print(f'\n--- {TASK_NAMES[task_id]} ---')
    print(f'{"Model":<25} {"wF1":>12} {"bAcc":>10} {"N":>5}')
    print('-' * 55)
    task_df = df[df['task'] == task_id]
    for m in sorted(task_df['model'].unique()):
        m_df = task_df[task_df['model'] == m]
        wf1 = m_df[METRIC].values
        bacc = m_df['balanced_accuracy'].values if 'balanced_accuracy' in m_df else np.array([])
        wf1_str = f'{np.mean(wf1)*100:.1f} +/- {np.std(wf1,ddof=1)*100:.1f}' if len(wf1)>1 else f'{np.mean(wf1)*100:.1f}'
        bacc_str = f'{np.mean(bacc)*100:.1f}' if len(bacc)>0 else '-'
        print(f'{m:<25} {wf1_str:>12} {bacc_str:>10} {len(wf1):>5}')

# --- Significance Tests ---
print('\n' + '=' * 70)
print('  SIGNIFICANCE TESTS: GEDA vs Baselines')
print('=' * 70)
for task_id in TASKS:
    geda = df[(df['model']=='geda_full') & (df['task']==task_id)][METRIC].values
    if len(geda) < 2:
        continue
    print(f'\n--- {TASK_NAMES[task_id]} ---')
    baselines = [m for m in df['model'].unique() if m != 'geda_full']
    for bl in baselines:
        bl_s = df[(df['model']==bl) & (df['task']==task_id)][METRIC].values
        if len(bl_s) < 2: continue
        t_stat, p_val = stats.ttest_ind(geda, bl_s, equal_var=False)
        pooled = np.sqrt((np.var(geda,ddof=1)+np.var(bl_s,ddof=1))/2)
        d = (np.mean(geda)-np.mean(bl_s))/(pooled+1e-8)
        n_tests = len(baselines)*len(TASKS)
        sig = 'YES***' if p_val < 0.05/n_tests else ('yes*' if p_val < 0.05 else 'no')
        delta = (np.mean(geda)-np.mean(bl_s))*100
        print(f'  GEDA vs {bl:<20} delta={delta:+.2f}pp  p={p_val:.4f}  d={d:.2f}  sig={sig}')

In [ ]:
# === 8.2 Comparison Plots ===
import os
import matplotlib.pyplot as plt, matplotlib
matplotlib.rcParams['figure.dpi'] = 120

df = pd.read_csv('/content/geda_results/all_results.csv')
METRIC = 'weighted_f1'
TASKS = ['task1', 'task2', 'task3']
TASK_NAMES = {'task1': 'Informativeness', 'task2': 'Humanitarian', 'task3': 'Damage'}
FEW_SHOTS = [50, 100, 250, 500]

paper1_candidates = sorted([m for m in df['model'].dropna().unique() if str(m).startswith('paper1_gnn')])
paper1_main = 'paper1_gnn_clip_pca' if 'paper1_gnn_clip_pca' in paper1_candidates else (paper1_candidates[0] if paper1_candidates else None)

MODELS_TO_PLOT = [m for m in [paper1_main, 'paper2_diffattn', 'geda_full'] if m is not None]
COLORS = {'paper2_diffattn': '#FF9800', 'geda_full': '#4CAF50'}
LABELS = {'paper2_diffattn': 'Paper 2 (DiffAttn)', 'geda_full': 'GEDA (Ours)'}
if paper1_main is not None:
    COLORS[paper1_main] = '#2196F3'
    LABELS[paper1_main] = f'Paper 1 (GNN: {paper1_main.replace("paper1_gnn_", "")})'

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, task_id in zip(axes, TASKS):
    task_df = df[df['task'] == task_id]
    for mn in MODELS_TO_PLOT:
        m_df = task_df[task_df['model'] == mn]
        if len(m_df) == 0:
            continue

        means, stds, vs = [], [], []
        has_few_shot = ('few_shot' in m_df.columns) and m_df['few_shot'].notna().any()

        if has_few_shot:
            for fs in FEW_SHOTS:
                sc = m_df[m_df['few_shot'] == fs][METRIC].dropna().values
                if len(sc) > 0:
                    means.append(np.mean(sc) * 100)
                    stds.append(np.std(sc, ddof=1) * 100 if len(sc) > 1 else 0)
                    vs.append(fs)
        else:
            sc = m_df[METRIC].dropna().values
            if len(sc) > 0:
                mean = np.mean(sc) * 100
                std = np.std(sc, ddof=1) * 100 if len(sc) > 1 else 0
                means = [mean] * len(FEW_SHOTS)
                stds = [std] * len(FEW_SHOTS)
                vs = FEW_SHOTS

        if len(means) > 0:
            ax.errorbar(vs, means, yerr=stds, marker='o', label=LABELS.get(mn, mn),
                        color=COLORS.get(mn, '#999'), linewidth=2, markersize=6, capsize=3)

    ax.set_title(TASK_NAMES[task_id], fontsize=13, fontweight='bold')
    ax.set_xlabel('Few-shot size')
    ax.set_xscale('log')
    ax.set_xticks(FEW_SHOTS)
    ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Weighted F1 (%)')
axes[0].legend(loc='lower right')
fig.suptitle('GEDA vs Baselines -- Few-shot Performance', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
os.makedirs('/content/geda_results/figures', exist_ok=True)
plt.savefig('/content/geda_results/figures/3model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure saved!')


In [ ]:
# === 8.3 LaTeX Table: 3-Model Comparison ===
import numpy as np

df = pd.read_csv('/content/geda_results/all_results.csv')
paper1_candidates = sorted([m for m in df['model'].dropna().unique() if str(m).startswith('paper1_gnn')])
paper1_main = 'paper1_gnn_clip_pca' if 'paper1_gnn_clip_pca' in paper1_candidates else (paper1_candidates[0] if paper1_candidates else None)

MODELS_ORDER = [m for m in [paper1_main, 'paper2_diffattn', 'geda_full'] if m is not None]
MODEL_NAMES = {
    'paper2_diffattn': 'Paper 2 (DiffAttn)',
    'geda_full': '\\textbf{GEDA}'
}
if paper1_main is not None:
    MODEL_NAMES[paper1_main] = 'Paper 1 (GNN)'

TASKS = ['task1', 'task2', 'task3']
TASK_LATEX = {'task1': 'Inform.', 'task2': 'Human.', 'task3': 'Damage'}
FEW_SHOTS = [50, 100, 250, 500]

for task_id in TASKS:
    print(f'\n% === {TASK_LATEX[task_id]} ===')
    print('\\begin{table}[h]')
    print('\\centering')
    print(f'\\caption{{Weighted F1 (\\%) \\textemdash\\ {TASK_LATEX[task_id]}}}')
    print('\\begin{tabular}{l' + ' c' * len(FEW_SHOTS) + '}')
    print('\\hline')
    hdr = '\\textbf{Model}'
    for fs in FEW_SHOTS:
        hdr += f' & \\textbf{{{fs}}}'
    print(hdr + ' \\\\')
    print('\\hline')

    for m in MODELS_ORDER:
        row = MODEL_NAMES.get(m, m)
        m_task = df[(df['model'] == m) & (df['task'] == task_id)]
        has_few_shot = ('few_shot' in m_task.columns) and m_task['few_shot'].notna().any()

        for fs in FEW_SHOTS:
            if has_few_shot:
                sc = m_task[m_task['few_shot'] == fs]['weighted_f1'].dropna().values
            else:
                sc = m_task['weighted_f1'].dropna().values

            if len(sc) > 1:
                row += f' & {np.mean(sc)*100:.1f} $\\pm$ {np.std(sc, ddof=1)*100:.1f}'
            elif len(sc) == 1:
                row += f' & {sc[0]*100:.1f}'
            else:
                row += ' & -'

        print(row + ' \\\\')

    print('\\hline')
    print('\\end{tabular}')
    print('\\end{table}')


In [ ]:
# === Download Results to Local Machine ===
import os, zipfile

try:
    from google.colab import files
except ImportError:
    files = None

RESULTS_DIR = '/content/geda_results'
ZIP_PATH = '/content/geda_results_all.zip'

# Zip everything
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk(RESULTS_DIR):
        for fname in fnames:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, '/content')
            zf.write(fpath, arcname)
            print(f'  + {arcname}')

size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f'\nZipped: {ZIP_PATH} ({size_mb:.1f} MB)')

if files is not None:
    print('Downloading...\n')
    files.download(ZIP_PATH)
else:
    print('google.colab not found. Download manually from:', ZIP_PATH)


# Phase 9: CausalCrisis Experiments

Load dataset from Drive if available, save results persistently to Drive, and run 60 rounds of CausalCrisis.

In [ ]:
# === 9.1 Mount Google Drive (if not already mounted) ===
try:
    from google.colab import drive
    import os
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
        print('Google Drive connected.')
    else:
        print('Google Drive already connected.')
except ImportError:
    print('Not in Colab. Skipping Drive mount.')


In [ ]:
# === 9.2 Upload Causal Models ===
import os

# In case the user uploads them via the sidebar:
if not os.path.exists('/content/causal_crisis_model.py'):
    print('Please upload causal_crisis_model.py and causal_crisis_trainer.py via Colab sidebar.')
else:
    print('[OK] causal_crisis files are present.')


In [ ]:
# === 9.3 Run 60 CausalCrisis Experiments ===
import os, sys
sys.path.insert(0, '/content')

if not os.path.exists('/content/causal_crisis_trainer.py'):
    print('Missing scripts! Stopping.')
else:
    from src.trainers.causal_crisis_trainer import run_causal_all_experiments
    
    # Auto-detect Drive for safe saving & loading
    DRIVE_DIR = '/content/drive/MyDrive/CrisisSummarization'
    RESULTS_CSV = '/content/causal_results/all_results.csv'
    DATASET = '/content/datasets/CrisisMMD_v2.0'
    
    if os.path.exists('/content/drive/MyDrive'):
        os.makedirs(f'{DRIVE_DIR}/causal_results', exist_ok=True)
        RESULTS_CSV = f'{DRIVE_DIR}/causal_results/all_results.csv'
        print(f'  [SAFE MODE] Saving results directly to Google Drive: {RESULTS_CSV}')
        
        drive_datasets = [
            '/content/drive/MyDrive/CrisisMMD_v2.0',
            '/content/drive/MyDrive/datasets/CrisisMMD_v2.0',
            f'{DRIVE_DIR}/CrisisMMD_v2.0'
        ]
        for d_path in drive_datasets:
            if os.path.exists(d_path):
                DATASET = d_path
                print(f'  [SAFE MODE] Loading dataset from Google Drive: {DATASET}')
                break
    else:
        print(f'  [WARNING] Google Drive not mounted. Saving locally.')
    
    run_causal_all_experiments(
        dataset_path=DATASET,
        seeds=(42, 123, 456, 789, 1024),
        tasks=('task1', 'task2', 'task3'),
        few_shot_sizes=(50, 100, 250, 500),
        device='cuda',
        results_csv=RESULTS_CSV,
        variant_name='causal_full',
        use_causal=True,
        use_intervention=True,
        use_causal_graph=True,
    )


# Phase 10: Comparison - GEDA vs CausalCrisis


In [ ]:
# === 10.1 CausalCrisis vs GEDA unified analysis ===
import pandas as pd
import numpy as np
from scipy import stats
import os

print('\n' + '='*70)
print('  PHASE 10: GEDA vs CausalCrisis COMPARISON')
print('='*70)

geda_csv = '/content/geda_results/all_results.csv'
if not os.path.exists(geda_csv) and os.path.exists('/content/drive/MyDrive'):
    drive_geda = '/content/drive/MyDrive/CrisisSummarization/geda_results/all_results.csv'
    if os.path.exists(drive_geda):
        geda_csv = drive_geda
        print(f'  [SAFE MODE] Loading GEDA results from: {geda_csv}')

causal_csv = RESULTS_CSV if 'RESULTS_CSV' in locals() else '/content/causal_results/all_results.csv'
if not os.path.exists(causal_csv) and os.path.exists('/content/drive/MyDrive'):
    drive_causal = '/content/drive/MyDrive/CrisisSummarization/causal_results/all_results.csv'
    if os.path.exists(drive_causal):
        causal_csv = drive_causal

dfs = {}
if os.path.exists(geda_csv):
    dfs['geda_full'] = pd.read_csv(geda_csv)
    print(f'  GEDA: {len(dfs["geda_full"])} runs')

if os.path.exists(causal_csv):
    dfs['causal_full'] = pd.read_csv(causal_csv)
    print(f'  CausalCrisis: {len(dfs["causal_full"])} runs')

if len(dfs) < 2:
    print('  [WARN] Need both result files to compare!')
else:
    df_all = pd.concat(dfs.values(), ignore_index=True)
    task_names = {'task1': 'Informativeness', 'task2': 'Humanitarian', 'task3': 'Damage'}

    print('\n' + '='*70)
    print('  COMPARISON TABLE: Weighted F1 (mean ± std)')
    print('='*70)
    for task in ['task1', 'task2', 'task3']:
        print(f'\n--- {task_names[task]} ---')
        print(f'{"Model":20s} {"N=50":>12s} {"N=100":>12s} {"N=250":>12s} {"N=500":>12s}')
        print('-' * 75)
        for model_name in ['geda_full', 'causal_full']:
            row = f'{model_name:20s}'
            for n in [50, 100, 250, 500]:
                mask = (df_all['model'] == model_name) & (df_all['task'] == task) & (df_all['few_shot'] == n)
                vals = df_all.loc[mask, 'weighted_f1'].values * 100
                if len(vals) > 0:
                    row += f'  {vals.mean():.1f}±{vals.std():.1f}'
                else:
                    row += '         N/A'
            print(row)
            
    print('\n' + '='*70)
    print('  SIGNIFICANCE TESTS')
    print('='*70)
    for task in ['task1', 'task2', 'task3']:
        print(f'\n--- {task_names[task]} ---')
        for n in [50, 100, 250, 500]:
            g = df_all.loc[(df_all['model'] == 'geda_full') & (df_all['task'] == task) & (df_all['few_shot'] == n), 'weighted_f1'].values
            c = df_all.loc[(df_all['model'] == 'causal_full') & (df_all['task'] == task) & (df_all['few_shot'] == n), 'weighted_f1'].values
            if len(g) >= 2 and len(c) >= 2:
                t, p = stats.ttest_ind(c, g)
                d = (c.mean() - g.mean()) * 100
                sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
                print(f'  N={n:3d}: diff={d:+.1f}% p={p:.4f} {sig}')

    if 'domain_acc_causal' in df_all.columns:
        cr = df_all[df_all['model'] == 'causal_full']
        if 'domain_acc_causal' in cr.columns:
            print('\n' + '='*70)
            print('  DOMAIN INVARIANCE CHECK (Target: ≈ 14.3%)')
            for task in ['task1', 'task2', 'task3']:
                vals = cr.loc[cr['task'] == task, 'domain_acc_causal'].values
                vals = vals[vals >= 0]
                if len(vals) > 0:
                    print(f'  {task_names[task]:20s}: {vals.mean()*100:.1f}%')
